# Preparación del dataset para modelado

Esta libreta ejecuta y audita `processing.preparacion_modelado` usando `agricultura_clima_edafologia_historial_2003_2025.csv`. El dataset integrado original no se modifica: se genera un producto separado con nombres canónicos, ausencias estructurales homologadas, banderas de calidad e historial recalculado con identificadores estables y sin fuga temporal.

In [1]:
import importlib

import numpy as np
import pandas as pd

from processing import preparacion_modelado as pm

pm = importlib.reload(pm)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


2026-11-24 17:41:55.908 | INFO     | proyecto_agronomia.config:<module>:14 - PROJ_ROOT path is: B:\MCD2\JW\agro\proyecto_agronomia


## Rutas y configuración

La ejecución completa puede tardar varios minutos porque lee y transforma todo el dataset nacional. Si el CSV ya fue generado, cambia `EJECUTAR_PREPARACION` a `False` para limitarte a recargarlo y revisar los controles.

In [2]:
RUTA_ENTRADA = pm.RUTA_ENTRADA
RUTA_SALIDA = pm.RUTA_SALIDA
EJECUTAR_PREPARACION = True
MINIMO_PERIODOS = 3

if not RUTA_ENTRADA.is_file():
    raise FileNotFoundError(f"No se encontró el dataset histórico: {RUTA_ENTRADA}")

columnas_entrada = pd.read_csv(RUTA_ENTRADA, nrows=0).columns.tolist()
faltantes_entrada = pm.COLUMNAS_REQUERIDAS - set(columnas_entrada)
assert not faltantes_entrada, (
    "La entrada no corresponde al integrado con historial; "
    f"faltan: {sorted(faltantes_entrada)}"
)

print(f"Entrada: {RUTA_ENTRADA}")
print(f"Salida:  {RUTA_SALIDA}")
print(f"Tamaño de entrada: {RUTA_ENTRADA.stat().st_size / 1024**2:,.1f} MB")
print(f"Columnas de entrada: {len(columnas_entrada):,}")


Entrada: B:\MCD2\JW\agro\proyecto_agronomia\data\processed\agricultura_clima_edafologia_historial_2003_2025.csv
Salida:  B:\MCD2\JW\agro\proyecto_agronomia\data\processed\agricultura_modelado_2003_2025.csv
Tamaño de entrada: 660.9 MB
Columnas de entrada: 101


## Generar o recargar el dataset

La función conserva todas las filas. La selección para entrenamiento se realiza posteriormente mediante `registro_apto_modelado`.

In [3]:
if EJECUTAR_PREPARACION:
    df_modelado = pm.generar_dataset_modelado(
        ruta_entrada=RUTA_ENTRADA,
        ruta_salida=RUTA_SALIDA,
        minimo_periodos=MINIMO_PERIODOS,
    )
else:
    if not RUTA_SALIDA.is_file():
        raise FileNotFoundError(
            "El dataset de modelado todavía no existe; activa EJECUTAR_PREPARACION."
        )
    df_modelado = pd.read_csv(
        RUTA_SALIDA, dtype={"cvegeo": "string"}, low_memory=False
    )

print(f"Registros cargados: {len(df_modelado):,}")
print(f"Columnas: {df_modelado.shape[1]:,}")
display(df_modelado.head())


Dataset de entrada: B:\MCD2\JW\agro\proyecto_agronomia\data\processed\agricultura_clima_edafologia_historial_2003_2025.csv
Dataset de modelado: B:\MCD2\JW\agro\proyecto_agronomia\data\processed\agricultura_modelado_2003_2025.csv
Registros conservados: 825,089
Registros con falla estricta de calidad: 6
Registros aptos para modelado: 803,737
Registros cargados: 825,089
Columnas: 113


,Anio,Idestado,Nomestado,Idddr,Nomddr,Idcader,Nomcader,Idmunicipio,Nommunicipio,Idciclo,Nomcicloproductivo,Idmodalidad,Nommodalidad,Idunidadmedida,Nomunidad,Idcultivo,Nomcultivo,Sembrada,Cosechada,Siniestrada,Volumenproduccion,Rendimiento,Preciomediorural,Valorproduccion,cvegeo,precipitacion_ciclo_mm,dias_clima_ciclo,temperatura_media_ciclo_c,temperatura_ene_c,temperatura_feb_c,temperatura_mar_c,temperatura_abr_c,temperatura_may_c,temperatura_jun_c,temperatura_jul_c,temperatura_ago_c,temperatura_sep_c,temperatura_oct_c,temperatura_nov_c,temperatura_dic_c,precipitacion_ene_mm,precipitacion_feb_mm,precipitacion_mar_mm,precipitacion_abr_mm,precipitacion_may_mm,precipitacion_jun_mm,precipitacion_jul_mm,precipitacion_ago_mm,precipitacion_sep_mm,precipitacion_oct_mm,precipitacion_nov_mm,precipitacion_dic_mm,temperatura_oct_anterior_c,temperatura_nov_anterior_c,temperatura_dic_anterior_c,precipitacion_oct_anterior_mm,precipitacion_nov_anterior_mm,precipitacion_dic_anterior_mm,mascara,grupo_suelo_dominante,grupo_suelo_dominante_pct,textura_dominante,textura_dominante_pct,grupos_suelo_n,nombre_suelo_wrb,suelo_pct_AC,suelo_pct_AL,suelo_pct_AN,suelo_pct_AR,suelo_pct_AT,suelo_pct_CH,suelo_pct_CL,suelo_pct_CM,suelo_pct_DU,suelo_pct_FL,suelo_pct_GL,suelo_pct_GY,suelo_pct_HS,suelo_pct_KS,suelo_pct_LP,suelo_pct_LV,suelo_pct_LX,suelo_pct_NT,suelo_pct_PH,suelo_pct_PL,suelo_pct_PT,suelo_pct_RG,suelo_pct_SA,suelo_pct_SC,suelo_pct_SN,suelo_pct_ST,suelo_pct_TC,suelo_pct_UM,suelo_pct_VR,area_mascara_ha,suelo_fuente,Nomestado_canonico,Nommunicipio_canonico,Nomcultivo_canonico,calidad_superficie,calidad_volumen,calidad_valor,calidad_rendimiento_fuente,registro_calidad_valida,sin_cosecha,sin_produccion,Rendimiento_modelado,registro_apto_modelado,rendimiento_hist_anios,rendimiento_anio_anterior,rendimiento_hist_media,rendimiento_hist_desviacion,rendimiento_hist_cv
0,2003,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Otoño-Invierno,1,Riego,200201,Tonelada,5490000,Avena forrajera en verde,1062.0,1062.0,0.0,28529.0,26.86,323.4,9226278.6,01001,84.8,182,13.835165,11.774194,13.915,14.978387,19.277,22.555161,20.925667,18.777742,18.896129,18.098667,16.434839,14.971667,11.090323,4.16,1.98,0.0,0.99,12.23,108.17,204.01,111.18,147.91,48.57,3.47,0.0,17.647097,13.39,11.299677,51.92,26.61,0.13,anual,PH,27.10,2.0,96.25,11,Phaeozem,0.71,0.0,0.0,0.0,0.0,0.0,10.87,17.39,5.23,1.70,0.0,0.0,0.0,1.27,7.67,0.0,0.0,0.0,27.10,24.31,0.0,0.0,0.25,0.0,0.0,0.0,3.5,0.0,0.0,75192.04,mascara,Aguascalientes,Aguascalientes,Avena forrajera en verde,True,True,True,True,True,False,False,26.863465,True,0.0,NaN,NaN,NaN,NaN
1,2003,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Otoño-Invierno,1,Riego,200201,Tonelada,5900000,Cebada forrajera en verde,2.0,2.0,0.0,54.0,27.00,250.0,13500.0,01001,84.8,182,13.835165,11.774194,13.915,14.978387,19.277,22.555161,20.925667,18.777742,18.896129,18.098667,16.434839,14.971667,11.090323,4.16,1.98,0.0,0.99,12.23,108.17,204.01,111.18,147.91,48.57,3.47,0.0,17.647097,13.39,11.299677,51.92,26.61,0.13,anual,PH,27.10,2.0,96.25,11,Phaeozem,0.71,0.0,0.0,0.0,0.0,0.0,10.87,17.39,5.23,1.70,0.0,0.0,0.0,1.27,7.67,0.0,0.0,0.0,27.10,24.31,0.0,0.0,0.25,0.0,0.0,0.0,3.5,0.0,0.0,75192.04,mascara,Aguascalientes,Aguascalientes,Cebada forrajera en verde,True,True,True,True,True,False,False,27.000000,True,0.0,NaN,NaN,NaN,NaN
2,2003,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Otoño-Invierno,1,Riego,200201,Tonelada,5980000,Centeno forrajero en verde,17.0,17.0,0.0,1020.0,60.00,350.0,357000.0,01001,84.8,182,13.835165,11.774194,13.915,14.978387,19.277,22.555161,20.925667,18.777742,18.896129,18.098667,16.434839,14.971667,11.090323,4.16,1.98,0.0,0.99,12.23,108.17,204.01,111.18,147.91,48.57,3.47,0.0,17.647097,13.39,11.299677,51.92,26.61,0.13,anual,PH,27.10,2.0,96.25,11,Phaeozem,0.71,0.0,0.0,0.0,0.0,0.0,10.87,17.39,5.23,1.70,0.0,0.0,0.0,1.27,7.67,0.0,0.0,0.0,27.10,24.31,0.0,0.0,0.25,0.0,0.0,0.0,3.5,0.0,0.0,75192.04,masc

## Integridad del producto

Se verifica que la transformación no haya perdido ni multiplicado registros y que el periodo y las claves municipales permanezcan completos.

In [4]:
registros_entrada = sum(1 for _ in open(RUTA_ENTRADA, encoding="utf-8")) - 1
columnas_nuevas = [
    "Nomestado_canonico", "Nommunicipio_canonico", "Nomcultivo_canonico",
    "calidad_superficie", "calidad_volumen", "calidad_valor",
    "calidad_rendimiento_fuente", "registro_calidad_valida",
    "sin_cosecha", "sin_produccion", "Rendimiento_modelado",
    "registro_apto_modelado", *pm.COLUMNAS_HISTORICAS,
]
faltantes = set(columnas_nuevas) - set(df_modelado.columns)
assert not faltantes, f"Faltan columnas de modelado: {sorted(faltantes)}"
assert len(df_modelado) == registros_entrada
assert df_modelado["cvegeo"].astype("string").str.zfill(5).str.fullmatch(r"\d{5}").all()
assert set(df_modelado["Anio"].unique()) == set(range(2003, 2026))
assert set(pm.COLUMNAS_INTEGRACION).issubset(df_modelado.columns)

resumen_integridad = pd.DataFrame({
    "registros_entrada": [registros_entrada],
    "registros_salida": [len(df_modelado)],
    "municipios": [df_modelado["cvegeo"].nunique()],
    "cultivos_id": [df_modelado["Idcultivo"].nunique()],
    "anio_inicial": [df_modelado["Anio"].min()],
    "anio_final": [df_modelado["Anio"].max()],
})
display(resumen_integridad)


,registros_entrada,registros_salida,municipios,cultivos_id,anio_inicial,anio_final
0,825089,825089,2452,416,2003,2025


## Banderas de calidad y elegibilidad

Una falla de calidad conserva el registro para auditoría, pero impide que se use para entrenar. La falta de cosecha o producción no es un error: simplemente significa que no existe un objetivo de rendimiento modelable.

In [5]:
columnas_banderas = [
    "calidad_superficie", "calidad_volumen", "calidad_valor",
    "calidad_rendimiento_fuente", "registro_calidad_valida",
    "sin_cosecha", "sin_produccion", "registro_apto_modelado",
]
resumen_banderas = pd.DataFrame({
    "verdaderos": df_modelado[columnas_banderas].sum(),
    "falsos": (~df_modelado[columnas_banderas]).sum(),
})
resumen_banderas["porcentaje_verdadero"] = (
    100 * resumen_banderas["verdaderos"] / len(df_modelado)
).round(4)
display(resumen_banderas)

columnas_revision = [
    "Anio", "cvegeo", "Nomestado", "Nommunicipio", "Nomcultivo",
    "Sembrada", "Cosechada", "Siniestrada", "Volumenproduccion",
    "Preciomediorural", "Valorproduccion", "calidad_superficie",
    "calidad_volumen", "calidad_valor",
]
display(
    df_modelado.loc[~df_modelado["registro_calidad_valida"], columnas_revision]
)


,verdaderos,falsos,porcentaje_verdadero
calidad_superficie,825086,3,99.9996
calidad_volumen,825087,2,99.9998
calidad_valor,825088,1,99.9999
calidad_rendimiento_fuente,825089,0,100.0000
registro_calidad_valida,825083,6,99.9993
sin_cosecha,21348,803741,2.5874
sin_produccion,21346,803743,2.5871
registro_apto_modelado,803737,21352,97.4122


,Anio,cvegeo,Nomestado,Nommunicipio,Nomcultivo,Sembrada,Cosechada,Siniestrada,Volumenproduccion,Preciomediorural,Valorproduccion,calidad_superficie,calidad_volumen,calidad_valor
244404,2009,26032,Sonora,Huásabas,Frijol,6.00,0.00,6.0,2.40,0.00,0.00,True,False,True
244412,2009,26067,Sonora,Villa Hidalgo,Frijol,38.00,0.00,38.0,15.20,0.00,0.00,True,False,True
609056,2019,22014,Querétaro,Querétaro,Tuna,20.00,20.00,15.0,40.00,1025.00,41000.00,False,True,True
769609,2024,18015,Nayarit,Santiago Ixcuintla,Plantero de tabaco,6.00,6.00,0.0,28119000.00,0.36,10260000.00,True,True,False
810046,2025,20168,Oaxaca,San José Estancia Grande,Pastos y praderas,1062.80,1063.14,0.0,14418.43,417.60,6021087.74,False,True,True
810212,2025,20485,Oaxaca,Santiago Tapextla,Pastos y praderas,1205.02,1206.63,0.0,15595.02,404.32,6305321.01,False,True,True


## Homologación de ausencias estructurales

Cuando no hay superficie cosechada, el rendimiento debe ser nulo. Cuando no existe producción, el precio tampoco representa una observación comercial.

In [6]:
assert df_modelado.loc[df_modelado["sin_cosecha"], "Rendimiento"].isna().all()
assert df_modelado.loc[df_modelado["sin_produccion"], "Preciomediorural"].isna().all()
assert df_modelado.loc[df_modelado["sin_cosecha"], "Rendimiento_modelado"].isna().all()

resumen_ausencias = (
    df_modelado.assign(
        rendimiento_nulo=df_modelado["Rendimiento"].isna(),
        precio_nulo=df_modelado["Preciomediorural"].isna(),
    )
    .groupby("Anio", as_index=False)
    .agg(
        registros=("cvegeo", "size"),
        sin_cosecha=("sin_cosecha", "sum"),
        sin_produccion=("sin_produccion", "sum"),
        rendimiento_nulo=("rendimiento_nulo", "sum"),
        precio_nulo=("precio_nulo", "sum"),
        aptos_modelado=("registro_apto_modelado", "sum"),
    )
)
display(resumen_ausencias)


,Anio,registros,sin_cosecha,sin_produccion,rendimiento_nulo,precio_nulo,aptos_modelado
0,2003,35667,1324,1324,1324,1324,34343
1,2004,36204,1266,1266,1266,1266,34938
2,2005,36520,1758,1758,1758,1758,34762
3,2006,36056,1379,1379,1379,1379,34677
4,2007,36136,1268,1268,1268,1268,34868
5,2008,35785,1209,1209,1209,1209,34576
6,2009,35562,1595,1593,1595,1593,33967
7,2010,36228,1328,1328,1328,1328,34900
8,2011,36299,1939,1939,1939,1939,34360
9,2012,36263,1043,1043,1043,1043,35220


## Nombres canónicos

Los nombres originales se conservan, pero cada código recibe la denominación más reciente para presentación y selección consistente.

In [7]:
cambios_nombre_cultivo = (
    df_modelado.loc[
        df_modelado["Nomcultivo"].ne(df_modelado["Nomcultivo_canonico"]),
        ["Idcultivo", "Nomcultivo", "Nomcultivo_canonico", "Anio"],
    ]
    .groupby(["Idcultivo", "Nomcultivo", "Nomcultivo_canonico"], as_index=False)
    .agg(anio_inicial=("Anio", "min"), anio_final=("Anio", "max"), registros=("Anio", "size"))
)
display(cambios_nombre_cultivo)


,Idcultivo,Nomcultivo,Nomcultivo_canonico,anio_inicial,anio_final,registros
0,5270000,Alpiste grano,Alpiste,2015,2019,18
1,5320000,Alpiste forrajero verde,Alpiste verde,2016,2018,2
2,7230000,Jenjibre,Jengibre,2003,2019,35
3,9250000,Semilla de zacate,Semilla de pastos y praderas,2015,2017,3


## Historial sin fuga temporal

Se inspecciona una serie con suficiente historia. Las variables del año actual solo deben resumir años anteriores; un hueco temporal deja `rendimiento_anio_anterior` nulo aunque existan observaciones más antiguas.

In [8]:
claves_historia = [*pm.CLAVES_HISTORIA_MODELO]
series = (
    df_modelado.loc[df_modelado["registro_apto_modelado"]]
    .groupby(claves_historia, observed=True)
    .agg(anios=("Anio", "nunique"), anio_inicial=("Anio", "min"), anio_final=("Anio", "max"))
    .sort_values("anios", ascending=False)
)
clave_ejemplo = series.index[0]
mascara_ejemplo = pd.Series(True, index=df_modelado.index)
for columna, valor in zip(claves_historia, clave_ejemplo):
    mascara_ejemplo &= df_modelado[columna].eq(valor)

columnas_historial = [
    "Anio", "Nomcultivo", "Nomcultivo_canonico", "Cosechada",
    "Volumenproduccion", "Rendimiento_modelado", *pm.COLUMNAS_HISTORICAS,
]
ejemplo_historial = (
    df_modelado.loc[mascara_ejemplo, columnas_historial]
    .sort_values("Anio")
)
display(pd.DataFrame([dict(zip(claves_historia, clave_ejemplo))]))
display(ejemplo_historial)

primeros = (
    df_modelado.loc[df_modelado["registro_apto_modelado"]]
    .sort_values([*claves_historia, "Anio"])
    .groupby(claves_historia, observed=True, group_keys=False)
    .head(1)
)
assert primeros["rendimiento_hist_anios"].eq(0).all()
assert primeros["rendimiento_anio_anterior"].isna().all()


,cvegeo,Idcultivo,Idciclo,Idmodalidad,Idunidadmedida
0,01001,6530000,3,1,200201


,Anio,Nomcultivo,Nomcultivo_canonico,Cosechada,Volumenproduccion,Rendimiento_modelado,rendimiento_hist_anios,rendimiento_anio_anterior,rendimiento_hist_media,rendimiento_hist_desviacion,rendimiento_hist_cv
21,2003,Durazno,Durazno,64.00,300.00,4.687500,0.0,NaN,NaN,NaN,NaN
35689,2004,Durazno,Durazno,79.00,839.00,10.620253,1.0,4.687500,NaN,NaN,NaN
71891,2005,Durazno,Durazno,76.00,1232.00,16.210526,2.0,10.620253,NaN,NaN,NaN
108410,2006,Durazno,Durazno,62.00,766.00,12.354839,3.0,16.210526,10.506093,5.762361,0.548478
144466,2007,Durazno,Durazno,71.00,674.00,9.492958,4.0,12.354839,10.968280,4.794894,0.437160
180602,2008,Durazno,Durazno,72.00,703.00,9.763889,5.0,9.492958,10.673215,4.204589,0.393938
216388,2009,Durazno,Durazno,72.00,698.00,9.694444,6.0,9.763889,10.521661,3.778977,0.359162
251949,2010,Durazno,Durazno,62.00,682.00,11.000000,7.0,9.694444,10.403487,3.463858,0.332952
288178,2011,Durazno,Durazno,92.00,1152.00,12.521739,8.0,11.000000,10.478051,3.213836,0.306721
324479,2012,Durazno,Durazno,94.00,1186.00,12.617021,9.0,12.521739,10.705128,3.082487,0.287945


## Cobertura disponible para entrenamiento

El catálogo se resume por identificador estable. Esta tabla permitirá decidir los mínimos de años y municipios en la notebook de entrenamiento.

In [9]:
catalogo_modelado = (
    df_modelado.loc[df_modelado["registro_apto_modelado"]]
    .groupby(["Idcultivo", "Nomcultivo_canonico", "Idunidadmedida"], as_index=False, observed=True)
    .agg(
        registros=("cvegeo", "size"),
        municipios=("cvegeo", "nunique"),
        anios=("Anio", "nunique"),
        superficie_cosechada=("Cosechada", "sum"),
        anio_inicial=("Anio", "min"),
        anio_final=("Anio", "max"),
    )
    .sort_values("superficie_cosechada", ascending=False)
)
display(catalogo_modelado.head(30))
print(f"Cultivos con registros aptos: {catalogo_modelado['Idcultivo'].nunique():,}")
print(f"Registros aptos: {df_modelado['registro_apto_modelado'].sum():,}")
print(f"Archivo generado: {RUTA_SALIDA}")


,Idcultivo,Nomcultivo_canonico,Idunidadmedida,registros,municipios,anios,superficie_cosechada,anio_inicial,anio_final
250,7490000,Maíz grano,200201,105079,2432,23,1.609257e+08,2003,2025
432,15050000,Pastos y praderas,200201,28762,1128,21,5.006176e+07,2003,2025
365,8810000,Sorgo grano,200201,23228,823,23,3.649216e+07,2003,2025
182,6840000,Frijol,200201,64932,2143,23,3.418383e+07,2003,2025
87,5820000,Caña de azúcar,200201,7707,328,23,1.702829e+07,2003,2025
76,5710000,Café cereza,200201,11207,513,23,1.597158e+07,2003,2025
391,9050000,Trigo grano,200201,15797,766,23,1.438753e+07,2003,2025
56,5490000,Avena forrajera en verde,200201,30392,1041,23,1.211470e+07,2003,2025
248,7470000,Maíz forrajero en verde,200201,14366,668,23,1.074463e+07,2003,2025
285,7850000,Naranja,200201,12666,604,23,7.570887e+06,2003,2025


Cultivos con registros aptos: 413
Registros aptos: 803,737
Archivo generado: B:\MCD2\JW\agro\proyecto_agronomia\data\processed\agricultura_modelado_2003_2025.csv
